# Demo 5 — Black-Box / Substitute Model Attack
## Transferability Attack via Model Substitution

**Threat Model:** Strict black-box — attacker only sees probability vectors

**Pipeline:**
1. Query victim model with synthetic inputs
2. Collect (input, soft-label) pairs
3. Train a **substitute model** on that data
4. Use substitute's gradients to craft adversarial examples
5. Transfer those examples to fool the **victim** (transferability)

**Key insight (Papernot et al., 2017):** Adversarial examples generalise
across models that learn similar decision boundaries.

In [ ]:
# Install
# !pip install torch torchvision matplotlib numpy

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
torch.manual_seed(0)
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.ToTensor()])
train_set_f = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set_f  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_set = torch.utils.data.Subset(train_set_f, range(1000))
test_set  = torch.utils.data.Subset(test_set_f, range(200))
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False)

## Step 1 — Train the Victim (Black Box) Model

In [ ]:
class VictimCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7,256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,10)
        )
    def forward(self, x): return self.net(x)

victim = VictimCNN().to(DEVICE)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
opt = optim.Adam(victim.parameters(), lr=1e-3)
fn  = nn.CrossEntropyLoss()
print('Training victim model...')
for epoch in range(3):
    victim.train()
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); fn(victim(X), y).backward(); opt.step()
victim.eval()
correct = sum((victim(X.to(DEVICE)).argmax(1)==y.to(DEVICE)).sum().item()
               for X, y in test_loader)
print(f'Victim accuracy: {correct/len(test_set)*100:.2f}%')

## Step 2 — Query the Black Box (API Simulation)

The attacker sends inputs and receives **soft probability vectors** — simulating
a real-world MLaaS API (e.g., Google Vision, AWS Rekognition).

In [ ]:
query_data   = torch.utils.data.Subset(train_set, range(1000))
query_loader = torch.utils.data.DataLoader(query_data, batch_size=256)
synthetic_inputs, synthetic_outputs = [], []
victim.eval()
with torch.no_grad():
    for X, _ in query_loader:
        X = X.to(DEVICE)
        soft_labels = torch.softmax(victim(X), dim=1)
        synthetic_inputs.append(X.cpu())
        synthetic_outputs.append(soft_labels.cpu())
synthetic_inputs  = torch.cat(synthetic_inputs)
synthetic_outputs = torch.cat(synthetic_outputs)
print(f'Collected {len(synthetic_inputs)} (input, soft-label) pairs from black-box API.')

## Step 3 — Train a Substitute Model

In [ ]:
class SubstituteCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self, x): return self.net(x)

substitute = SubstituteCNN().to(DEVICE)
sub_opt    = optim.Adam(substitute.parameters(), lr=1e-3)
kl_loss    = nn.KLDivLoss(reduction='batchmean')
syn_ds     = torch.utils.data.TensorDataset(synthetic_inputs, synthetic_outputs)
syn_loader = torch.utils.data.DataLoader(syn_ds, batch_size=64, shuffle=True)
print('Training substitute model on queried data...')
for epoch in range(10):
    substitute.train()
    for X, y_soft in syn_loader:
        X, y_soft = X.to(DEVICE), y_soft.to(DEVICE)
        sub_opt.zero_grad()
        kl_loss(torch.log_softmax(substitute(X), dim=1), y_soft).backward()
        sub_opt.step()
    if (epoch+1) % 5 == 0: print(f'  Epoch {epoch+1}/10 done.')

## Step 4 — Transfer Attack: Craft Adversarial Examples on Substitute, Evaluate on Victim

In [ ]:
def fgsm(model, X, y, epsilon):
    Xa = X.clone().detach().requires_grad_(True)
    fn(model(Xa), y).backward()
    return (X + epsilon * Xa.grad.sign()).clamp(0, 1).detach()

X_test, y_test = next(iter(test_loader))
X_test, y_test = X_test.to(DEVICE), y_test.to(DEVICE)
X_adv_sub = fgsm(substitute, X_test, y_test, epsilon=0.2)

victim.eval(); substitute.eval()
with torch.no_grad():
    acc_clean  = (victim(X_test).argmax(1) == y_test).float().mean().item()
    acc_attack = (victim(X_adv_sub).argmax(1) == y_test).float().mean().item()
    p_clean = victim(X_test).argmax(1)
    p_attack_sub = substitute(X_adv_sub).argmax(1)
    p_attack_vic = victim(X_adv_sub).argmax(1)

def save_images(imgs, preds, title, filename):
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
    for col in range(8):
        axes[col].imshow(imgs[col].squeeze().cpu(), cmap='gray')
        c = 'green' if preds[col]==y_test[col] else 'red'
        axes[col].set_title(f'P:{preds[col]}', color=c, fontsize=10)
        axes[col].axis('off')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()

save_images(X_test[:8], p_clean[:8], 'Before Attack (Clean)', '05_before_attack.png')
save_images(X_adv_sub[:8], p_attack_sub[:8], 'In Process (Adv on Substitute)', '05_in_process_attack.png')
save_images(X_adv_sub[:8], p_attack_vic[:8], 'After Attack (Evaluated on Victim)', '05_after_attack.png')

print(f'Victim accuracy on CLEAN inputs    : {acc_clean*100:.2f}%')
print(f'Victim accuracy on TRANSFERRED adv : {acc_attack*100:.2f}%')
print(f'Transfer drop                      : {(acc_clean-acc_attack)*100:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Clean', 'After Transfer Attack'],
              [acc_clean*100, acc_attack*100], color=['#27ae60', '#e74c3c'],
              edgecolor='black', width=0.4)
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylim(0, 110); ax.set_ylabel('Victim Accuracy (%)')
ax.set_title('Black-Box Transfer Attack\n(Substitute -> Victim)', fontweight='bold')
plt.tight_layout()
plt.savefig('05_blackbox_substitute_results.png', dpi=150)
plt.show()

## Summary

The substitute model attack shows **gradient access is not required** for
powerful adversarial examples. ~1000 API queries is enough to build a
surrogate and dramatically reduce victim accuracy via transferability.

**Defenses:** Query rate limiting, output perturbation, defensive distillation.